<a href="https://colab.research.google.com/github/h77k/python-ai-Trunova-Polina/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2: Data Analysis — Чтение и проверка данных (Skoda Auto Dataset)

**Цель:** Научиться читать CSV-файлы из личного репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
*   `data/skoda-auto.csv` — технические характеристики автомобилей Skoda: модель, годы производства, габариты (высота, ширина, длина), масса и ссылки.

**Структура данных:**
*   `model`, `modelLabel` — идентификатор и название модели.
*   `startProduction` — год начала производства.
*   `height`, `width`, `length`, `mass` — физические параметры автомобиля.
*   `website`, `image` — ссылки на источники.

**Что мы делаем:**
1.  Клонируем личный репозиторий GitHub в Colab.
2.  Читаем CSV-файл `skoda-auto.csv` в pandas DataFrame.
3.  Очищаем и переименовываем столбцы для удобства анализа.
4.  Смотрим структуру данных и делаем быструю валидацию типов.

## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Клонируем личный репозиторий в Colab

import os

# Имя вашего репозитория
repo = "python-ai-Trunova-Polina"
repo_url = "https://github.com/h77k/python-ai-Trunova-Polina.git"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Trunova-Polina
    !git clone -q {repo_url}

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Trunova-Polina
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Trunova-Polina


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [4]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas

import pandas as pd

# Загружаем основной файл с данными об автомобилях
df_skoda = pd.read_csv("data/skoda-auto.csv")

print("✅ Загружено строк в df_skoda:", len(df_skoda))
print("\nПервые 5 строк:")
df_skoda.head()

✅ Загружено строк в df_skoda: 198

Первые 5 строк:


,model,modelLabel,startProduction,predecessorLabel,successorLabel,height,width,length,mass,website,image
0,http://www.wikidata.org/entity/Q10403322,Q10403322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...
1,http://www.wikidata.org/entity/Q10404063,Q10404063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,http://www.wikidata.org/entity/Q10404082,Q10404082,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,http://www.wikidata.org/entity/Q10404317,Q10404317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...
4,http://www.wikidata.org/entity/Q10404862,Q10404862,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...


# 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле `skoda-auto.csv` есть технические столбцы и столбцы с читаемыми подписями (суффикс `Label`), которые удобнее использовать для анализа:

1.  **Столбец `model`** может содержать ID или ссылку, а **`modelLabel`** — понятное название модели. Мы переименуем `modelLabel` в `model` для удобства, а исходный `model` переименуем в `model_id` (или сохраним как есть, если он нужен).
2.  **Числовые столбцы:** `startProduction` (год), `height`, `width`, `length`, `mass` могут быть считаны как объекты или содержать пропуски. Их нужно привести к числовому типу (`int` или `float`).

**В этом шаге мы:**

*   Переименуем `modelLabel` → `model` (чтобы работать с названиями).
*   При необходимости переименуем исходный `model` → `model_id` (чтобы не потерять идентификатор).
*   Приведём столбцы `startProduction`, `height`, `width`, `length`, `mass` к числовому типу.

**При приведении к числам мы используем:**

*   `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`.
*   `.fillna(0)` или `.dropna()` — обработка пропущенных значений (в данном примере заменим на 0 для простоты агрегации, но в реальном анализе лучше удалять строки с пропусками в ключевых полях).
*   `.astype(int)` или `.astype(float)` — перевод столбца к нужному типу.

⚠️ **Важно:** Этот шаг обязателен для корректного построения графиков и статистического анализа (например, расчета средней массы или длины автомобиля).

In [5]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для Skoda Auto

# Создаем копию, чтобы не менять исходные данные случайно, или работаем напрямую
# Проверка: есть ли столбец modelLabel, требующий обработки?
if "modelLabel" in df_skoda.columns:

    # 1. Переименование столбцов
    # Сохраняем исходный ID модели, если он есть, чтобы не потерять
    if "model" in df_skoda.columns:
        df_skoda = df_skoda.rename(columns={"model": "model_id"})

    # Переименовываем читаемое название в основное поле 'model'
    df_skoda = df_skoda.rename(columns={
        "modelLabel": "model",
    })

    # 2. Приведение числовых столбцов к правильному типу
    numeric_cols = ['startProduction', 'height', 'width', 'length', 'mass']

    for col in numeric_cols:
        if col in df_skoda.columns:
            # to_numeric превратит ошибки в NaN
            df_skoda[col] = pd.to_numeric(df_skoda[col], errors='coerce')

            # Опционально: заполняем пропуски нулями или средним.
            # Для года производства и габаритов замена на 0 может исказить статистику,
            # но для демонстрации метода оставляем fillna(0) как в шаблоне.
            # В серьезном анализе лучше использовать: df_skoda.dropna(subset=[col])
            df_skoda[col] = df_skoda[col].fillna(0).astype(int) # или float, если нужны дроби

    print("✅ df_skoda очищен и преобразован")
    print(f"   Переименованы столбцы. Числовые колонки обработаны: {numeric_cols}")
else:
    print("⏭️ df_skoda уже очищен (нет столбца modelLabel), пропускаем переименование")

# Показать итоговую структуру
print("\n✅ Данные готовы к анализу")
print(df_skoda.dtypes)
print("\nПервые 5 строк после очистки:")
df_skoda.head()

✅ df_skoda очищен и преобразован
   Переименованы столбцы. Числовые колонки обработаны: ['startProduction', 'height', 'width', 'length', 'mass']

✅ Данные готовы к анализу
model_id            object
model               object
startProduction      int64
predecessorLabel    object
successorLabel      object
height               int64
width                int64
length               int64
mass                 int64
website             object
image               object
dtype: object

Первые 5 строк после очистки:


,model_id,model,startProduction,predecessorLabel,successorLabel,height,width,length,mass,website,image
0,http://www.wikidata.org/entity/Q10403322,Q10403322,0,NaN,NaN,0,0,0,0,NaN,http://commons.wikimedia.org/wiki/Special:File...
1,http://www.wikidata.org/entity/Q10404063,Q10404063,0,NaN,NaN,0,0,0,0,NaN,NaN
2,http://www.wikidata.org/entity/Q10404082,Q10404082,0,NaN,NaN,0,0,0,0,NaN,NaN
3,http://www.wikidata.org/entity/Q10404317,Q10404317,0,NaN,NaN,0,0,0,0,NaN,http://commons.wikimedia.org/wiki/Special:File...
4,http://www.wikidata.org/entity/Q10404862,Q10404862,0,NaN,NaN,0,0,0,0,NaN,http://commons.wikimedia.org/wiki/Special:File...


# 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор нашего DataFrame `df_skoda`:

1.  посмотрим размер таблицы (shape);
2.  выведем список столбцов;
3.  посмотрим первые несколько строк;
4.  дополнительно посчитаем базовую статистику по числовым параметрам (масса, габариты).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы структурировать вывод информации.

In [6]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_skoda, "Данные об автомобилях Skoda (df_skoda)")

# Быстрый взгляд на статистику по числовым колонкам
print("\n📈 Базовая статистика по числовым полям:")
print(df_skoda.describe())


📊 Данные об автомобилях Skoda (df_skoda)
Размер: (198, 11)
Столбцы: model_id, model, startProduction, predecessorLabel, successorLabel, height, width, length, mass, website, image

Первые строки:
                                   model_id      model  startProduction  \
0  http://www.wikidata.org/entity/Q10403322  Q10403322                0   
1  http://www.wikidata.org/entity/Q10404063  Q10404063                0   
2  http://www.wikidata.org/entity/Q10404082  Q10404082                0   
3  http://www.wikidata.org/entity/Q10404317  Q10404317                0   
4  http://www.wikidata.org/entity/Q10404862  Q10404862                0   

  predecessorLabel successorLabel  height  width  length  mass website  \
0              NaN            NaN       0      0       0     0     NaN   
1              NaN            NaN       0      0       0     0     NaN   
2              NaN            NaN       0      0       0     0     NaN   
3              NaN            NaN       0      0       0

# ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

*   сколько уникальных моделей автомобилей представлено в данных;
*   диапазон годов начала производства (`startProduction`);
*   какие модели встречаются чаще всего (если в данных есть разные модификации одной модели);
*   базовые характеристики: средняя масса, длина, ширина и высота.

**Метод `value_counts()`:**

*   считает, сколько раз каждое значение встречается в столбце;
*   сортирует результаты по убыванию.

Например, `df_skoda["model"].value_counts().head()` покажет Топ-5 самых часто встречающихся названий моделей в датасете.

**Метод `describe()`:**

*   показывает минимальное, максимальное, среднее значение и квантили для числовых столбцов, что помогает быстро выявить аномалии (например, отрицательную массу или год производства в будущем).

In [7]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных Skoda Auto")

# 1. Общая информация по категориям
print("\n📊 Анализ категорий:")
print("Уникальных моделей (model):", df_skoda["model"].nunique())
print("Уникальных предшественников (predecessorLabel):", df_skoda["predecessorLabel"].nunique())
print("Уникальных преемников (successorLabel):", df_skoda["successorLabel"].nunique())

# 2. Анализ временного ряда (Год начала производства)
if "startProduction" in df_skoda.columns:
    min_year = df_skoda["startProduction"].min()
    max_year = df_skoda["startProduction"].max()
    print(f"\n📅 Диапазон годов начала производства: {int(min_year)} — {int(max_year)}")

    # Топ-5 годов с наибольшим количеством новых моделей
    print("\nТоп-5 годов с наибольшим числом запусков моделей:")
    print(df_skoda["startProduction"].value_counts().head())

# 3. Самые популярные модели в датасете (по количеству строк/модификаций)
print("\n🚗 Топ-5 моделей по количеству записей в датасете:")
print(df_skoda["model"].value_counts().head())

# 4. Статистика по габаритам и массе
print("\n📏 Статистика по габаритам и массе:")
numeric_cols = ['height', 'width', 'length', 'mass']
# Выбираем только те колонки, которые есть в dataframe
existing_numeric_cols = [col for col in numeric_cols if col in df_skoda.columns]

if existing_numeric_cols:
    print(df_skoda[existing_numeric_cols].describe())
else:
    print("⚠️ Числовые столбцы габаритов не найдены или не были преобразованы.")

🔍 Быстрая проверка данных Skoda Auto

📊 Анализ категорий:
Уникальных моделей (model): 181
Уникальных предшественников (predecessorLabel): 36
Уникальных преемников (successorLabel): 31

📅 Диапазон годов начала производства: 0 — 0

Топ-5 годов с наибольшим числом запусков моделей:
startProduction
0    198
Name: count, dtype: int64

🚗 Топ-5 моделей по количеству записей в датасете:
model
Škoda Rapid              5
Škoda Superb III         4
Škoda Tudor (2002)       3
Škoda Fabia RS Rally2    2
Škoda 125                2
Name: count, dtype: int64

📏 Статистика по габаритам и массе:
            height        width        length         mass
count   198.000000   198.000000    198.000000   198.000000
mean    335.464646   475.949495   1145.363636   106.313131
std     667.969357   786.264301   2140.917553   720.267106
min       0.000000     0.000000      0.000000     0.000000
25%       0.000000     0.000000      0.000000     0.000000
50%       0.000000     0.000000      0.000000     0.000000
75

# 📝 Summary

Что мы сделали в этом ноутбуке (Week 2):

✅ Клонировали ваш репозиторий `python-ai-Trunova-Polina` в Colab
✅ Прочитали CSV-файл `data/skoda-auto.csv`
✅ Переименовали столбцы (`modelLabel` → `model`) и привели числовые данные (год, габариты, масса) к правильному типу
✅ Проверили структуру данных (размер, столбцы, первые строки)
✅ Выполнили быструю валидацию:
   *   количество уникальных моделей
   *   диапазон годов производства
   *   топ популярных моделей
   *   базовая статистика по габаритам и массе

Теперь у нас есть аккуратный, проверенный DataFrame `df_skoda`, с которым удобно работать дальше.

**Планы на следующую неделю:**

В следующем ноутбуке мы перейдем к более глубокому анализу, соответствующему требованиям вашего проекта:

1.  **Анализ временных рядов:** Построение ряда по количеству выпущенных моделей или изменению средних габаритов/массы по годам (`startProduction`).
2.  **Проверка стационарности:** Использование теста Дики-Фуллера (ADF test).
3.  **Визуализация:** Построение графиков динамики характеристик автомобилей во времени.
4.  **Эконометрическое моделирование:** Подготовка данных для построения модели ARIMA или регрессионного анализа.

🎯 *Цель: подготовить чистые данные и первичные графики для отчета по индивидуальному проекту.*